# dllm on Colab

Masked diffusion LM (ModernBERT) 프리트레인 -> 샘플링 -> SFT -> 인퍼런스.

로컬 RTX 4060 Ti(8GB) 실측이 **4,300 tok/s** 였습니다. Colab GPU별 대략치:

| GPU | VRAM | precision | 예상 tok/s |
|---|---|---|---|
| T4 (무료) | 16GB | fp16 (bf16 미지원) | ~6k |
| L4 | 24GB | bf16 | ~15k |
| A100 40GB | 40GB | bf16 | ~50k |

**런타임 유형을 GPU로 바꾸고 시작하세요.** 아래 셀을 위에서부터 순서대로 실행하면 됩니다.

## 1. GPU 확인

In [1]:
!nvidia-smi

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

/bin/bash: line 1: nvidia-smi: command not found
torch 2.11.0+cpu | cuda False


## 2. 의존성 설치

ModernBERT는 transformers 4.48 이상이 필요합니다. 설치 후 **런타임 재시작 요구가 뜨면 재시작**하고
이 셀 아래부터 다시 실행하세요.

In [ ]:
!pip install -q -U "transformers>=4.48" datasets accelerate safetensors tokenizers wandb

## 3. 코드 가져오기 (git clone)

코드를 고칠 때마다 로컬에서 push 하고 여기서 이 셀만 다시 실행하면 됩니다.

리포가 비공개라면 URL에 토큰을 끼워 넣어야 하는데, 노트북에 토큰이 그대로 남지 않게 이렇게 받으세요:

```python
from getpass import getpass
token = getpass("GitHub token: ")
REPO_URL = f"https://{token}@github.com/Jaehyeon-kr/DLLM.git"
```

In [10]:
%pwd

UsageError: CWD no longer exists - please use %cd to change directory.


In [13]:
!rm -rf /content/dllm
!git clone --depth 1 https://github.com/Jaehyeon-kr/DLLM.git /content/dllm
!echo "--- exit code: $? ---"
!ls -la /content/dllm

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/dllm'...
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
--- exit code: 0 ---
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
ls: cannot access '/content/dllm': No such file or directory


import os
os.chdir("/content")          # 살아있는 폴더로 탈출
!rm -rf /content/dllm
!git clone --depth 1 https://github.com/Jaehyeon-kr/DLLM.git /content/dllm
!ls /content/dllm

In [64]:
REPO_URL = "https://github.com/Jaehyeon-kr/DLLM.git"

import os, shutil

CODE_DIR = "/content/dllm"

### 이미 받아둔 게 있으면 지우고 새로 받는다. 매 세션 최신을 쓰는 게 헷갈릴 일이 없다 ###
if os.path.exists(CODE_DIR):
    shutil.rmtree(CODE_DIR)

!git clone --depth 1 {REPO_URL} {CODE_DIR}

print(sorted(f for f in os.listdir(CODE_DIR) if f.endswith(".py")))

Cloning into '/content/dllm'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 20 (delta 4), reused 10 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 57.75 KiB | 14.44 MiB/s, done.
Resolving deltas: 100% (4/4), done.
['data_utils.py', 'finetune_sft.py', 'inference.py', 'prepare_data.py', 'prepare_data_sft.py', 'pretrain.py', 'sample.py', 'tokenizer.py']


### Google Drive 마운트 (체크포인트 보관용)

코드는 git에서 받더라도 체크포인트는 Drive에 두는 게 좋습니다. Colab 세션이 끊기면 `/content`는
전부 날아갑니다. Drive를 안 쓸 거면 이 셀은 건너뛰고 다음 셀의 `WORK_DIR`을 `/content/dllm_exp`로
바꾸세요.

In [28]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 4. 설정

GPU를 보고 precision과 배치를 자동으로 잡습니다.

`per_gpu_batch_size`가 **실효 배치**이고 `gradient_accumulation_steps`로 나눈 값이 실제 미니배치입니다
(`pretrain.py`의 `mini_batchsize = per_gpu_batch_size // gradient_accumulation_steps`).
아래는 실효 배치를 128로 고정하고, VRAM이 허락하는 만큼 미니배치를 키워 accum을 줄이는 방식입니다.

In [4]:
import os, torch, math

### 체크포인트는 Drive에 두는 게 안전합니다. 세션이 끊겨도 남습니다 ###
WORK_DIR   = "/content/drive/MyDrive/dllm_exp"   # Drive를 안 쓴다면 "/content/dllm_exp"
DATA_DIR   = "/content/prepped_data"             # 재생성이 싸니 로컬 디스크로 충분
SFT_DATA_DIR = "/content/prepped_sft_data"

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3

### T4는 Turing이라 bf16이 없습니다. 이 경우 fp16으로 떨어뜨립니다 ###
MIXED_PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"

### seq 1024 기준으로 대략 이 정도가 안전선입니다 ###
if   vram_gb >= 38: MINI_BATCH = 16
elif vram_gb >= 22: MINI_BATCH = 8
elif vram_gb >= 15: MINI_BATCH = 4
else:               MINI_BATCH = 2

EFFECTIVE_BATCH = 128
ACCUM = max(1, EFFECTIVE_BATCH // MINI_BATCH)

### C4 샤드 하나가 약 170M 토큰 / 디스크 330MB 입니다 ###
NUM_C4_SHARDS = 4

print(f"gpu             : {gpu_name} ({vram_gb:.1f} GB)")
print(f"mixed_precision : {MIXED_PRECISION}")
print(f"mini batch      : {MINI_BATCH}")
print(f"effective batch : {EFFECTIVE_BATCH}  (accum {ACCUM})")
print(f"tokens / step   : {EFFECTIVE_BATCH * 1024 / 1000:.0f}k")

os.environ["PYTHONPATH"] = CODE_DIR
os.chdir(CODE_DIR)

gpu             : NVIDIA A100-SXM4-40GB (39.5 GB)
mixed_precision : bf16
mini batch      : 16
effective batch : 128  (accum 8)
tokens / step   : 131k


## 5. 프리트레인 데이터 준비

C4를 받아서 토크나이즈하고 1024 길이로 패킹합니다. 샤드 4개면 15~25분 정도 걸립니다.

In [5]:
!python prepare_data.py \
    --test_split_pct 0.005 \
    --context_length 1024 \
    --path_to_data_store {DATA_DIR} \
    --dataset_split_seed 42 \
    --num_workers 4 \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --num_c4_shards {NUM_C4_SHARDS}

config.json: 1.19kB [00:00, 3.57MB/s]
tokenizer_config.json: 20.8kB [00:00, 57.6MB/s]
tokenizer.json: 2.13MB [00:00, 129MB/s]
special_tokens_map.json: 100% 694/694 [00:00<00:00, 4.00MB/s]
README.md: 41.1kB [00:00, 87.1MB/s]
en/c4-train.00002-of-01024.json.gz: 100% 320M/320M [00:03<00:00, 89.8MB/s]
en/c4-train.00000-of-01024.json.gz: 100% 319M/319M [00:04<00:00, 64.3MB/s]
en/c4-train.00003-of-01024.json.gz: 100% 319M/319M [00:06<00:00, 51.8MB/s]
en/c4-train.00001-of-01024.json.gz: 100% 318M/318M [00:06<00:00, 47.1MB/s]
Generating train split: 1425269 examples [00:09, 150844.78 examples/s]
Map (num_proc=4):   0% 0/1418142 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (8708 > 8192). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (10567 > 8192). Running this sequence through the model will result

## 6. 스텝 수 정하기

먼저 10스텝만 돌려서 이 GPU의 실제 s/step을 재고, 주어진 시간 예산에 맞는 `num_training_steps`를
역산합니다. Colab 세션은 보통 12시간에서 끊기니 예산은 그보다 짧게 잡으세요.

In [7]:
import time, subprocess

PROBE_STEPS = 10
start = time.time()
subprocess.run([
    "python", "pretrain.py",
    "--experiment_name", "probe",
    "--working_directory", "/content/probe",
    "--hf_model_name", "answerdotai/ModernBERT-base",
    "--mixed_precision", MIXED_PRECISION,
    "--path_to_prepped_data", DATA_DIR,
    "--num_workers", "2",
    "--per_gpu_batch_size", str(EFFECTIVE_BATCH),
    "--gradient_accumulation_steps", str(ACCUM),
    "--num_training_steps", str(PROBE_STEPS),
    "--num_warmup_steps", "1",
    "--evaluation_interval", "10000",
    "--checkpoint_interval", "10000",
    "--logging_steps", "10000",
    "--max_grad_norm", "1.0",
    "--learning_rate", "1e-4",
    "--no-log_wandb",
], check=True)
elapsed = time.time() - start

### 모델 로딩/컴파일 오버헤드가 섞여 있으니 넉넉히 30초를 빼줍니다 ###
sec_per_step = max(0.1, (elapsed - 30) / PROBE_STEPS)
tok_per_sec  = EFFECTIVE_BATCH * 1024 / sec_per_step

BUDGET_HOURS = 8
NUM_TRAINING_STEPS = int(BUDGET_HOURS * 3600 / sec_per_step)

print(f"\n{sec_per_step:.2f} s/step  |  {tok_per_sec:,.0f} tok/s")
print(f"{BUDGET_HOURS}시간 예산 -> num_training_steps = {NUM_TRAINING_STEPS:,}")
print(f"총 학습 토큰 {NUM_TRAINING_STEPS * EFFECTIVE_BATCH * 1024 / 1e9:.2f}B "
      f"(코퍼스 대비 {NUM_TRAINING_STEPS * EFFECTIVE_BATCH * 1024 / (NUM_C4_SHARDS * 170e6):.1f} 에폭)")


6.07 s/step  |  21,584 tok/s
8시간 예산 -> num_training_steps = 4,742
총 학습 토큰 0.62B (코퍼스 대비 0.9 에폭)


## 7. 프리트레인

- `max_grad_norm`은 반드시 **1.0**입니다. 원본 `pretrain.sh`에 있던 1e-4는 오타이고, 그 값이면 매 업데이트가
  1만분의 1로 줄어들어 학습이 사실상 멈춥니다.
- `num_warmup_steps`는 전체의 1~2% 정도가 무난합니다.
- `checkpoint_interval`을 너무 촘촘히 잡지 마세요. `accelerator.save_state`는 옵티마이저까지 저장해서
  한 번에 1.8GB 가까이 씁니다. Drive 용량이 순식간에 찹니다.
- `evaluation_interval`도 마찬가지입니다. eval 한 번이 수십 초에서 수 분입니다.

In [32]:
WARMUP  = max(10, NUM_TRAINING_STEPS // 50)     # 2%
EVAL_IV = max(50, NUM_TRAINING_STEPS // 20)     # 20번
CKPT_IV = max(50, NUM_TRAINING_STEPS // 10)     # 10개
LOG_IV  = max(10, NUM_TRAINING_STEPS // 200)

print(f"warmup {WARMUP} | eval every {EVAL_IV} | ckpt every {CKPT_IV} | log every {LOG_IV}")

!python ./dllm/pretrain.py \
    --experiment_name "dllm" \
    --working_directory {WORK_DIR} \
    --resume_from_checkpoint {WORK_DIR}/dllm/checkpoint_536 \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --mixed_precision {MIXED_PRECISION} \
    --path_to_prepped_data {DATA_DIR} \
    --num_workers 2 \
    --per_gpu_batch_size {EFFECTIVE_BATCH} \
    --gradient_accumulation_steps {ACCUM} \
    --num_training_steps {NUM_TRAINING_STEPS} \
    --max_grad_norm 1.0 \
    --lr_scheduler_type cosine \
    --num_warmup_steps {WARMUP} \
    --logging_steps {LOG_IV} \
    --evaluation_interval {EVAL_IV} \
    --checkpoint_interval {CKPT_IV} \
    --learning_rate 1e-4 \
    --weight_decay 2e-5 \
    --no-log_wandb

warmup 94 | eval every 237 | ckpt every 474 | log every 23
Loading weights: 100% 137/137 [00:00<00:00, 2120.39it/s, Materializing param=model.layers.21.mlp_norm.weight]     
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
Number of Parameters: 149659077
Resuming from checkpoint /content/drive/MyDrive/dllm_exp/dllm/checkpoint_536 at step 536
[552/4742] Training Loss: 3.8535869121551514]
[575/4742] Training Loss: 3.6441171169281006]
[598/4742] Training Loss: 3.3147459030151367]
[621/4742] Training Loss: 3.135668992996216]
[644/4742] Training Loss: 3.3935537338256836]
[667/4742] Training Loss: 3.3055810928344727]
[690/4742] Training Loss: 3.516462564468384]
Evaluating Model!!                      
 15% 711/4742 [07:31<2:53:11,  2.58s/it]
  0% 0/209 [00:00<?, ?it/s]
  0% 1/209 [00:00<00:

## 7-1. 프리트레인 이어서 (resume)

세션이 끊겨 위 셀이 중간에 멈췄다면 여기서 이어서 돌립니다. `WORK_DIR`을 Drive로 잡아뒀다면
가장 최근 `checkpoint_*`이 남아 있고, `--resume_from_checkpoint auto`가 그중 최신 것을 골라
**모델·옵티마이저·스케줄러·RNG 상태까지 복원**한 뒤 그 스텝부터 계속합니다.

**주의 — 아래 인자는 위 7번 셀과 전부 같아야 합니다.** 특히:

- `--num_training_steps` : cosine 스케줄이 이 값 기준으로 그려지므로 원래 값(`NUM_TRAINING_STEPS`)과
  달라지면 LR이 어긋납니다. 더 오래 돌리고 싶다면 처음부터 큰 값으로 다시 시작해야 합니다.
- `--per_gpu_batch_size`, `--gradient_accumulation_steps`, `--learning_rate`, `--num_warmup_steps` 등도 동일하게.

이어서 저장되는 체크포인트는 원래 `checkpoint_interval`(= `CKPT_IV`) 간격 그대로 찍힙니다.
복원 직후 첫 스텝에서 방금 읽은 체크포인트를 한 번 다시 저장(그리고 eval)하는데, 무해합니다.

특정 체크포인트를 지정하려면 `auto` 대신 디렉토리 경로를 주세요:
`--resume_from_checkpoint {WORK_DIR}/dllm/checkpoint_536`

In [ ]:
### 세션이 끊겼다 다시 붙었다면: 1~4번 셀(GPU/설치/clone/설정)을 먼저 다시 실행해서
### WORK_DIR, MIXED_PRECISION, EFFECTIVE_BATCH, ACCUM 이 정의돼 있어야 합니다.
###
### num_training_steps 는 원래 학습에 쓴 값과 반드시 같아야 합니다. probe 셀(6번)을 다시
### 돌리면 타이밍 노이즈로 값이 조금 달라질 수 있으니, 원래 값을 여기에 직접 박아 둡니다.
NUM_TRAINING_STEPS = 5368   # <- 원래 7번 셀에서 쓴 값으로 맞추세요

### 아래 인자는 7번 셀과 동일해야 합니다 (스케줄/배치 일관성) ###
WARMUP  = max(10, NUM_TRAINING_STEPS // 50)
EVAL_IV = max(50, NUM_TRAINING_STEPS // 20)
CKPT_IV = max(50, NUM_TRAINING_STEPS // 10)
LOG_IV  = max(10, NUM_TRAINING_STEPS // 200)

### 어디서부터 이어지는지 미리 확인 ###
import glob
_ck = sorted(glob.glob(f"{WORK_DIR}/dllm/checkpoint_*"),
             key=lambda p: int(p.rsplit("_", 1)[1]))
print("found checkpoints:", [os.path.basename(p) for p in _ck])
print("resuming from    :", os.path.basename(_ck[-1]) if _ck else "(none!)")

!python pretrain.py \
    --experiment_name "dllm" \
    --working_directory {WORK_DIR} \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --mixed_precision {MIXED_PRECISION} \
    --path_to_prepped_data {DATA_DIR} \
    --num_workers 2 \
    --per_gpu_batch_size {EFFECTIVE_BATCH} \
    --gradient_accumulation_steps {ACCUM} \
    --num_training_steps {NUM_TRAINING_STEPS} \
    --max_grad_norm 1.0 \
    --lr_scheduler_type cosine \
    --num_warmup_steps {WARMUP} \
    --logging_steps {LOG_IV} \
    --evaluation_interval {EVAL_IV} \
    --checkpoint_interval {CKPT_IV} \
    --learning_rate 1e-4 \
    --weight_decay 2e-5 \
    --resume_from_checkpoint auto \
    --no-log_wandb

## 8. 샘플링

`--show_steps`를 주면 스텝마다 `_`(아직 안 채워진 위치)가 줄어드는 게 보입니다.
`--path_to_checkpoint`는 디렉토리를 받습니다 (`sample.py`는 그 안에서 model.safetensors를 찾습니다).

In [44]:
!python ./dllm/sample.py \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --path_to_checkpoint /content/drive/MyDrive/dllm_exp/dllm/checkpoint_4740 \
    --prompt "The Paris is" \
    --gen_length 64 \
    --num_steps 128 \
    --block_length 32 \
    --strategy confidence \
    --temperature 0.7 \
    --top_p 0.9 \
    --num_samples 1 \
    --seed 42 \
    --show_steps

Loading weights: 100% 137/137 [00:00<00:00, 1868.17it/s, Materializing param=model.layers.21.mlp_norm.weight]     
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
Loaded /content/drive/MyDrive/dllm_exp/dllm/checkpoint_4740/model.safetensors
  missing keys: 1 (first few: ['decoder.weight'])
Denoising... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00 -:--:--
Traceback (most recent call last):
  File "/content/./dllm/sample.py", line 399, in <module>
    main()
  File "/content/./dllm/sample.py", line 383, in main
    x, prompt_length = generate(model, tokenizer, args, device)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
  

In [43]:
!ls  

dllm  drive  prepped_data  prepped_sft_data  probe  sample_data


In [47]:
import glob, os

ckpts = sorted(glob.glob(f"{WORK_DIR}/dllm/checkpoint_*"),
               key=lambda p: int(p.rsplit("_", 1)[1]))
CKPT = ckpts[-1]
print("using", CKPT)

!python ./dllm/sample.py \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --path_to_checkpoint {CKPT} \
    --prompt "Harry Potter" \
    --gen_length 128 \
    --num_steps 128 \
    --block_length 32 \
    --strategy confidence \
    --temperature 1.0 \
    --top_p 0.95 \
    --num_samples 2 \
    --seed 42 \
    --show_steps

using /content/drive/MyDrive/dllm_exp/dllm/checkpoint_4740
Loading weights: 100% 137/137 [00:00<00:00, 1077.06it/s, Materializing param=model.layers.21.mlp_norm.weight]     
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
Loaded /content/drive/MyDrive/dllm_exp/dllm/checkpoint_4740/model.safetensors
  missing keys: 1 (first few: ['decoder.weight'])
Denoising... ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   3% 0:00:00 0:00:04
t=0.875  masked=248
<BOS>Harry Potter is 
the[MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK
][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][
MASK] Potter[MASK] 
the[MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK
][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MASK][MA

## 9. SFT 데이터 준비 (Alpaca)

52k 샘플이라 1~2분이면 끝납니다.

In [ ]:
!git clone 

In [57]:
!python ./dllm/prepare_data_sft.py \
    --test_split_pct 0.01 \
    --context_length 1024 \
    --path_to_data_store {SFT_DATA_DIR} \
    --dataset_split_seed 42 \
    --num_workers 4 \
    --hf_model_name "answerdotai/ModernBERT-base"

Map (num_proc=4): 100% 51481/51481 [00:09<00:00, 5663.92 examples/s]
Map (num_proc=4): 100% 521/521 [00:00<00:00, 840.99 examples/s]
Number of Samples In Dataset: 51481
Filter (num_proc=4): 100% 51481/51481 [00:00<00:00, 64437.02 examples/s]
Filter (num_proc=4): 100% 521/521 [00:00<00:00, 2471.19 examples/s]
Number of Samples After Length Filter: 51481
Map (num_proc=4): 100% 51481/51481 [00:03<00:00, 16755.42 examples/s]
Map (num_proc=4): 100% 521/521 [00:00<00:00, 951.77 examples/s] 
Saving the dataset (1/1 shards): 100% 51481/51481 [00:00<00:00, 291760.08 examples/s]
Saving the dataset (1/1 shards): 100% 521/521 [00:00<00:00, 113253.82 examples/s]
Time to Load Dataset 0.005362749099731445
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'query_mask'],
        num_rows: 51481
    })
    test: Dataset({
        features: ['input_ids', 'query_mask'],
        num_rows: 521
    })
})


## 10. SFT

프리트레인과 달리 데이터가 51,481개로 고정이라 에폭 기준으로 스텝을 잡습니다. 보통 2~3에폭입니다.
LR도 프리트레인보다 한 자릿수 낮춥니다.

`--path_to_pretrained_checkpoint`는 디렉토리가 아니라 **`.safetensors` 파일 경로**입니다
(`finetune_sft.py`가 `load_file`로 직접 읽습니다).

In [58]:
from datasets import load_from_disk
d = load_from_disk(SFT_DATA_DIR)
print(d)
print("features:", d["train"].features)
row = d["train"][0]
print("keys:", list(row.keys()))
print("input_ids type:", type(row["input_ids"]))
print("input_ids sample:", row["input_ids"] if not isinstance(row["input_ids"], list) else row["input_ids"][:10])


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'query_mask'],
        num_rows: 51481
    })
    test: Dataset({
        features: ['input_ids', 'query_mask'],
        num_rows: 521
    })
})
features: {'input_ids': List(Value('int32')), 'query_mask': List(Value('int64'))}
keys: ['input_ids', 'query_mask']
input_ids type: <class 'list'>
input_ids sample: [50368, 50370, 4537, 50371, 187, 37923, 253, 3451, 3159, 432]


In [59]:
from datasets import load_from_disk
d = load_from_disk(SFT_DATA_DIR)
print("input_ids type:", type(d["train"][0]["input_ids"]))


input_ids type: <class 'list'>


In [60]:
!grep -c return_dict /content/dllm/prepare_data_sft.py

2


In [61]:
SFT_EPOCHS = 3
SFT_TRAIN_SAMPLES = 51481
SFT_STEPS = SFT_EPOCHS * SFT_TRAIN_SAMPLES // EFFECTIVE_BATCH

PRETRAINED = f"{CKPT}/model.safetensors"

### ! 매직 안의 {} 치환은 단순한 이름일 때 가장 안전하니 미리 계산해 둡니다 ###
SFT_WARMUP  = max(10, SFT_STEPS // 50)
SFT_LOG_IV  = max(10, SFT_STEPS // 100)
SFT_EVAL_IV = max(50, SFT_STEPS // 10)
SFT_CKPT_IV = max(50, SFT_STEPS // 5)

print(f"{SFT_STEPS} steps from {PRETRAINED}")

!python ./dllm/finetune_sft.py \
    --experiment_name "dllm_sft" \
    --working_directory {WORK_DIR} \
    --path_to_pretrained_checkpoint {PRETRAINED} \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --mixed_precision {MIXED_PRECISION} \
    --path_to_prepped_data {SFT_DATA_DIR} \
    --num_workers 2 \
    --per_gpu_batch_size {EFFECTIVE_BATCH} \
    --gradient_accumulation_steps {ACCUM} \
    --num_training_steps {SFT_STEPS} \
    --max_grad_norm 1.0 \
    --lr_scheduler_type cosine \
    --num_warmup_steps {SFT_WARMUP} \
    --logging_steps {SFT_LOG_IV} \
    --evaluation_interval {SFT_EVAL_IV} \
    --checkpoint_interval {SFT_CKPT_IV} \
    --learning_rate 1e-5 \
    --weight_decay 2e-5 \
    --no-log_wandb

1206 steps from /content/drive/MyDrive/dllm_exp/dllm/checkpoint_4740/model.safetensors
Loading weights: 100% 137/137 [00:00<00:00, 3136.14it/s, Materializing param=model.layers.21.mlp_norm.weight]     
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
Number of Parameters: 149659077
[0/1206] Training Loss: 3.6843514442443848
Evaluating Model!!          
  0% 0/1206 [00:01<?, ?it/s]
  0% 0/33 [00:00<?, ?it/s]
  9% 3/33 [00:00<00:01, 24.11it/s]
 18% 6/33 [00:00<00:00, 27.20it/s]
 30% 10/33 [00:00<00:00, 26.53it/s]
 42% 14/33 [00:00<00:00, 28.48it/s]
 52% 17/33 [00:00<00:00, 28.75it/s]
 61% 20/33 [00:00<00:00, 28.86it/s]
 70% 23/33 [00:00<00:00, 28.04it/s]
 79% 26/33 [00:00<00:00, 27.91it/s]
 91% 30/33 [00:01<00:00, 28.82it/s]
100% 33/33 [00:01<00:00, 27.75it/s]
[0/1206] Validation Loss: 

In [41]:
import glob
print(glob.glob("/content/drive/MyDrive/dllm_exp/dllm/*"))

['/content/drive/MyDrive/dllm_exp/dllm/checkpoint_0', '/content/drive/MyDrive/dllm_exp/dllm/checkpoint_536', '/content/drive/MyDrive/dllm_exp/dllm/checkpoint_948', '/content/drive/MyDrive/dllm_exp/dllm/checkpoint_1422', '/content/drive/MyDrive/dllm_exp/dllm/checkpoint_1896', '/content/drive/MyDrive/dllm_exp/dllm/checkpoint_2370', '/content/drive/MyDrive/dllm_exp/dllm/checkpoint_2844', '/content/drive/MyDrive/dllm_exp/dllm/checkpoint_3318', '/content/drive/MyDrive/dllm_exp/dllm/checkpoint_3792', '/content/drive/MyDrive/dllm_exp/dllm/checkpoint_4266', '/content/drive/MyDrive/dllm_exp/dllm/checkpoint_4740', '/content/drive/MyDrive/dllm_exp/dllm/final_model']


## 11. 인퍼런스

SFT한 모델에 대화 템플릿을 씌워서 물어봅니다. `--strategy`는 `random` / `low_confidence` 두 개뿐이고,
`sample.py`의 `confidence`와는 이름만 다른 별개 구현입니다.

In [65]:
!python ./dllm/inference.py \
    --safetensors_path /content/drive/MyDrive/dllm_exp/dllm_sft/checkpoint_1205/model.safetensors \
    --seq_len 256 \
    --num_steps 256 \
    --strategy low_confidence \
    --hf_model_name "answerdotai/ModernBERT-base" \
    --prompt "What is artificial intelligence?"


Loading weights: 100% 137/137 [00:00<00:00, 765.89it/s, Materializing param=model.layers.21.mlp_norm.weight]      
Generating... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00 -:--:--
Traceback (most recent call last):
  File "/content/./dllm/inference.py", line 249, in <module>
    inference(input_tokens, 
    ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/content/./dllm/inference.py", line 126, in inference
    with Live("", refresh_per_second=5, console=console) as live:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/rich/live.py", line 166, in __enter__
    self.start(refresh=self._renderable is not None)
  File "/usr/local/lib/python3.12/dist-packages/rich/live.py", line 113, in start
    self.console.set_live(self)
  File "/usr/local/lib/python3.12/dist-packag

## 참고 — 세션이 끊겼을 때

`WORK_DIR`을 Drive로 잡아뒀다면 체크포인트가 남아 있습니다. **7-1번 셀**에서
`--resume_from_checkpoint auto`로 가장 최근 체크포인트를 읽어 그 스텝부터 이어서 돌릴 수 있습니다
(모델·옵티마이저·스케줄러·RNG까지 복원).

이어 돌릴 때 `--num_training_steps`는 원래 값과 같아야 cosine 스케줄이 어긋나지 않습니다.
데이터 로더 순서(어떤 배치를 봤는지)까지 복원되지는 않지만, 프리트레인은 데이터가 충분히 커서
실질적인 영향은 없습니다.

## 참고 — 세션이 끊겼을 때

`WORK_DIR`을 Drive로 잡아뒀다면 체크포인트가 남아 있습니다. 다만 `pretrain.py`에는 재개(resume) 경로가
없어서, 이어서 돌리려면 `finetune_sft.py`처럼 체크포인트를 초기 가중치로 읽는 방식으로 쓰거나
`accelerator.load_state`를 추가해야 합니다.